<a href="https://colab.research.google.com/github/LiweiWang-Serena/cs7374-assignment1-data-pipeline/blob/main/CS7374_Assignment_1_Data_Pipeline_for_Foundation_Model_Pretraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS7374 Assignment 1: Data Pipeline for Foundation Model Pretraining

 This notebook builds a complete pretraining data pipeline in five steps:

| Step | What it does | Output |
|---|---|---|
| 1. Data collection | Streams ~1.2 GB of text from three domains (Wikipedia, news, web) | `raw_*.jsonl` |
| 2. Cleaning | Strips HTML/markdown/citation markers, normalizes text, removes short and duplicate documents | `cleaned.jsonl` |
| 3. Tokenization | GPT-2 BPE tokenization and chunking into 512-token blocks | `tokenized_blocks.pt`, `sample_data/sample_dataset.pt` |
| 4. Data loader | Map-style and streaming PyTorch datasets with batching | batches of `input_ids` / `attention_mask` / `labels` |
| 5. Optional analysis | Domain coverage and document-length statistics | printed report |


## 0. Setup

In [7]:
!pip install -q datasets huggingface_hub transformers torch

In [8]:
from pathlib import Path

# Step 1: data collection
TARGET_GB = 1.2                 # total raw text to collect (>= 1 GB per assignment spec)
OUT_DIR = Path("./data")

# Step 2: cleaning
MIN_WORDS = 50                  # drop documents shorter than this

# Step 3: tokenization
TOKENIZER_NAME = "gpt2"
BLOCK_SIZE = 512                # max sequence length per training example
STRIDE = None                   # None = non-overlapping chunks
N_SAMPLE_BLOCKS = 10            # size of the sample file for submission

# Step 4: data loader
BATCH_SIZE = 8

# File paths
CLEANED_PATH = OUT_DIR / "cleaned.jsonl"
TOKENIZED_PATH = OUT_DIR / "tokenized_blocks.pt"
SAMPLE_PATH = Path("./sample_data/sample_dataset.pt")

OUT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_PATH.parent.mkdir(parents=True, exist_ok=True)

## 1. Data Collection

Three sources are used to satisfy the diversity requirement (Section 3):

| Domain | Dataset | Share of budget |
|---|---|---|
| Wikipedia (encyclopedic) | `wikimedia/wikipedia` (config `20231101.en`) | 60% |
| News | `vblagoje/cc_news` | 25% |
| General web text | `Skylion007/openwebtext` | 15% |


In [9]:
import json
from datasets import load_dataset


def bytes_of(text: str) -> int:
    """UTF-8 byte size of a string (used to track how close we are to the target size)."""
    return len(text.encode("utf-8"))


def collect_domain(dataset_name, split, text_field, domain_label, target_bytes, out_path,
                   config_name=None, streaming=True):
    """Stream a HF dataset and write raw documents (one JSON object per line)
    to out_path until target_bytes of raw text has been collected."""
    print(f"[collect] domain={domain_label} dataset={dataset_name} target={target_bytes/1e6:.1f}MB")
    ds = load_dataset(dataset_name, config_name, split=split, streaming=streaming)

    collected_bytes, n_docs = 0, 0
    with open(out_path, "w", encoding="utf-8") as f:
        for example in ds:
            text = example.get(text_field, "")
            if not text or not isinstance(text, str):
                continue
            f.write(json.dumps({"text": text, "domain": domain_label}, ensure_ascii=False) + "\n")
            collected_bytes += bytes_of(text)
            n_docs += 1
            if collected_bytes >= target_bytes:
                break

    print(f"[done] {domain_label}: {n_docs} docs, {collected_bytes/1e6:.1f}MB -> {out_path}")
    return n_docs, collected_bytes

In [10]:
total_target_bytes = int(TARGET_GB * (1024 ** 3))
split_ratio = {"wikipedia": 0.6, "news": 0.25, "web": 0.15}

# (domain, dataset_name, config_name, out_filename, fallback_hint)
domain_specs = [
    ("wikipedia", "wikimedia/wikipedia", "20231101.en", "raw_wikipedia.jsonl",
     'fallback: load_dataset("omarkamali/wikipedia-monthly", "latest.en", streaming=True)'),
    ("news", "vblagoje/cc_news", None, "raw_news.jsonl",
     "fallback: check the dataset's discussion page on the Hugging Face Hub"),
    ("web", "Skylion007/openwebtext", None, "raw_web.jsonl",
     'fallback: load_dataset("vietgpt/openwebtext_en", streaming=True)'),
]

manifest, failures = {}, []
for domain_label, dataset_name, config_name, out_filename, fallback_hint in domain_specs:
    try:
        n, b = collect_domain(
            dataset_name=dataset_name, config_name=config_name, split="train",
            text_field="text", domain_label=domain_label,
            target_bytes=int(total_target_bytes * split_ratio[domain_label]),
            out_path=OUT_DIR / out_filename,
        )
        manifest[domain_label] = {"docs": n, "bytes": b, "source": dataset_name}
    except Exception as e:  # one failing source should not stop the others
        print(f"\n[FAILED] domain={domain_label} dataset={dataset_name}\n  error: {e}\n  {fallback_hint}")
        failures.append(domain_label)

with open(OUT_DIR / "collection_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

total_bytes = sum(v["bytes"] for v in manifest.values())
print(f"\n=== TOTAL COLLECTED: {total_bytes/1e9:.3f} GB across {len(manifest)}/3 domains ===")
if failures:
    print(f"=== {len(failures)} domain(s) FAILED: {failures} -- see fallback hints above ===")

[collect] domain=wikipedia dataset=wikimedia/wikipedia target=773.1MB


README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

[done] wikipedia: 174374 docs, 773.1MB -> data/raw_wikipedia.jsonl
[collect] domain=news dataset=vblagoje/cc_news target=322.1MB


README.md:   0%|          | 0.00/7.57k [00:00<?, ?B/s]

[done] news: 144080 docs, 322.1MB -> data/raw_news.jsonl
[collect] domain=web dataset=Skylion007/openwebtext target=193.3MB


README.md:   0%|          | 0.00/7.46k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

[done] web: 38717 docs, 193.3MB -> data/raw_web.jsonl

=== TOTAL COLLECTED: 1.288 GB across 3/3 domains ===


## 2. Cleaning and Normalization

Cleaning steps (Section 4.1):

| Step | Tool |
|---|---|
| Strip HTML tags, markdown syntax (links, headers, emphasis), and citation markers such as `[12]` | `re` (regular expressions) |
| Remove irrelevant symbols (keeping basic punctuation) and collapse whitespace | `re` |
| Lowercase | `str.lower()` |
| Drop documents shorter than `MIN_WORDS` words | `str.split()` + `len()` |
| Remove duplicates by hashing the **cleaned** text, so documents that differ only in formatting, whitespace, or case are also caught | `hashlib` (SHA-256) + `set` |

In [11]:
import hashlib
import re

HTML_TAG_RE = re.compile(r"<[^>]+>")
MARKDOWN_LINK_RE = re.compile(r"\[([^\]]+)\]\((?:[^)]+)\)")   # [text](url) -> text
MARKDOWN_EMPHASIS_RE = re.compile(r"[*_]{1,3}")                  # *bold*, _italic_
MARKDOWN_HEADER_RE = re.compile(r"^#{1,6}\s*", flags=re.MULTILINE)
REFERENCE_MARKER_RE = re.compile(r"\[\d+\]")                    # citation markers like [12]
MULTI_WHITESPACE_RE = re.compile(r"\s+")
IRRELEVANT_SYMBOLS_RE = re.compile(r"[^\w\s.,!?;:'\"()%-]")      # keep basic punctuation


def clean_text(text: str) -> str:
    """Apply the full normalization pipeline to a single document."""
    text = HTML_TAG_RE.sub(" ", text)
    text = MARKDOWN_LINK_RE.sub(r"\1", text)
    text = MARKDOWN_HEADER_RE.sub("", text)
    text = MARKDOWN_EMPHASIS_RE.sub("", text)
    text = REFERENCE_MARKER_RE.sub("", text)
    text = IRRELEVANT_SYMBOLS_RE.sub(" ", text)
    text = text.lower()
    return MULTI_WHITESPACE_RE.sub(" ", text).strip()


def doc_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def word_count(text: str) -> int:
    return len(text.split())


def clean_file(in_path, seen_hashes, min_words=MIN_WORDS):
    """Generator that yields cleaned records from a single raw JSONL file."""
    kept = dropped_short = dropped_dup = dropped_empty = 0
    with open(in_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                dropped_empty += 1
                continue

            cleaned = clean_text(record.get("text", ""))
            if not cleaned:
                dropped_empty += 1
                continue
            if word_count(cleaned) < min_words:
                dropped_short += 1
                continue
            h = doc_hash(cleaned)
            if h in seen_hashes:
                dropped_dup += 1
                continue
            seen_hashes.add(h)
            kept += 1
            yield {"text": cleaned, "domain": record.get("domain", "unknown"), "hash": h}

    print(f"[clean] {Path(in_path).name}: kept={kept}, dropped_short={dropped_short}, "
          f"dropped_duplicate={dropped_dup}, dropped_empty_or_malformed={dropped_empty}")

Quick check of the cleaning function on a small example:

In [12]:
example = "<p>## Title\nThis is **bold** text with a [link](https://example.com) and a citation [12].</p>"
print("Before:", example)
print("After: ", clean_text(example))

Before: <p>## Title
This is **bold** text with a [link](https://example.com) and a citation [12].</p>
After:  title this is bold text with a link and a citation .


In [13]:
raw_files = sorted(OUT_DIR.glob("raw_*.jsonl"))
if not raw_files:
    raise SystemExit(f"No raw_*.jsonl files found in {OUT_DIR}. Run Step 1 first.")

seen_hashes, n_written = set(), 0
with open(CLEANED_PATH, "w", encoding="utf-8") as out_f:
    for raw_path in raw_files:
        for record in clean_file(raw_path, seen_hashes, MIN_WORDS):
            out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
            n_written += 1

print(f"\n=== Cleaning complete: {n_written} documents written to {CLEANED_PATH} ===")

[clean] raw_news.jsonl: kept=112970, dropped_short=17006, dropped_duplicate=14104, dropped_empty_or_malformed=0
[clean] raw_web.jsonl: kept=38714, dropped_short=1, dropped_duplicate=2, dropped_empty_or_malformed=0
[clean] raw_wikipedia.jsonl: kept=165380, dropped_short=8994, dropped_duplicate=0, dropped_empty_or_malformed=0

=== Cleaning complete: 317064 documents written to data/cleaned.jsonl ===


## 3. Tokenization

- **Tokenizer:** GPT-2 BPE (`TOKENIZER_NAME = "gpt2"`, `vocab_size = 50257`), a well-tested tokenizer compatible with most GPT-style causal language models. Change `TOKENIZER_NAME` to use another tokenizer, e.g. WordPiece via `"bert-base-uncased"`.
- **Chunking:** each document is split into non-overlapping blocks of `BLOCK_SIZE = 512` tokens (`STRIDE = None`). A tail chunk shorter than `BLOCK_SIZE // 4 = 128` tokens is dropped only if the document already produced a full chunk, so short documents are never lost.
- **Padding:** blocks are padded to 512 tokens with an `attention_mask` (1 = real token, 0 = padding). GPT-2 has no pad token, so EOS (`<|endoftext|>`, id 50256) is reused as pad.
- **Storage:** `input_ids` are stored as `int32` and `attention_mask` as `bool` to reduce file size (~1.6 GB instead of ~5.4 GB with `int64`); they are cast to `int64` in the data loader.
- **Output:** `tokenized_blocks.pt` with shape `(656119, 512)`, plus a 10-block sample in `sample_data/sample_dataset.pt`.


In [14]:
import torch
from transformers import AutoTokenizer


def chunk_tokens(token_ids, block_size, stride=None):
    """Split a list of token ids into fixed-size blocks (overlapping if stride < block_size)."""
    stride = stride or block_size
    chunks = []
    for start in range(0, len(token_ids), stride):
        chunk = token_ids[start:start + block_size]
        # Drop a too-small chunk only if it is a leftover tail of a longer document
        if chunks and len(chunk) < block_size // 4:
            break
        chunks.append(chunk)
        if start + block_size >= len(token_ids):
            break
    return chunks

In [15]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = int(1e12)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [16]:
import os, json, gc
import numpy as np
import torch
from tqdm.auto import tqdm

MAX_LINES = None
BATCH = 1000

with open(CLEANED_PATH, "rb") as f:
    total_lines = sum(1 for _ in f)
print(f"[tokenize] total lines: {total_lines}")

all_blocks, domain_counts = [], {}

def process(texts, domains):
    for ids, domain in zip(tokenizer(texts)["input_ids"], domains):
        blocks = chunk_tokens(ids, BLOCK_SIZE, STRIDE)
        all_blocks.extend(np.asarray(b, dtype=np.int32) for b in blocks)
        domain_counts[domain] = domain_counts.get(domain, 0) + len(blocks)

texts, domains = [], []
with open(CLEANED_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(tqdm(f, total=total_lines, desc="tokenizing")):
        if MAX_LINES and i >= MAX_LINES:
            break
        record = json.loads(line)
        texts.append(record["text"])
        domains.append(record.get("domain", "unknown"))
        if len(texts) >= BATCH:
            process(texts, domains)
            texts, domains = [], []
if texts:
    process(texts, domains)

print(f"[tokenize] total blocks: {len(all_blocks)}")
print(f"[tokenize] blocks per domain: {domain_counts}")

N = len(all_blocks)
pad_id = tokenizer.pad_token_id
padded = torch.full((N, BLOCK_SIZE), pad_id, dtype=torch.int32)
attention_mask = torch.zeros((N, BLOCK_SIZE), dtype=torch.bool)
for i, b in enumerate(tqdm(all_blocks, desc="padding")):
    padded[i, :len(b)] = torch.from_numpy(b)
    attention_mask[i, :len(b)] = True
del all_blocks
gc.collect()

os.makedirs(os.path.dirname(TOKENIZED_PATH) or ".", exist_ok=True)
torch.save({"input_ids": padded, "attention_mask": attention_mask}, TOKENIZED_PATH)
print(f"[tokenize] saved -> {TOKENIZED_PATH}  shape={tuple(padded.shape)}")

n = min(N_SAMPLE_BLOCKS, N)
torch.save({"input_ids": padded[:n].clone(), "attention_mask": attention_mask[:n].clone()}, SAMPLE_PATH)
print(f"[tokenize] saved sample ({n} blocks) -> {SAMPLE_PATH}")

[tokenize] total lines: 317064


tokenizing:   0%|          | 0/317064 [00:00<?, ?it/s]

[tokenize] total blocks: 656119
[tokenize] blocks per domain: {'news': 165254, 'web': 91937, 'wikipedia': 398928}


padding:   0%|          | 0/656119 [00:00<?, ?it/s]

[tokenize] saved -> data/tokenized_blocks.pt  shape=(656119, 512)
[tokenize] saved sample (10 blocks) -> sample_data/sample_dataset.pt


Inspect the first block by decoding it back to text:

In [17]:
data = torch.load(SAMPLE_PATH)
first_block = data["input_ids"][0][data["attention_mask"][0].bool()]
print(tokenizer.decode(first_block)[:500], "...")
del data

there's a surprising twist to regina willoughby's last season with columbia city ballet: it's also her 18-year-old daughter melina's first season with the company. regina, 40, will retire from the stage in march, just as her daughter starts her own career as a trainee. but for this one season, they're sharing the stage together. performing side-by-side in the nutcracker regina and melina are not only dancing in the same nutcracker this month, they're onstage at the same time: regina is doing sno ...


## 4. Custom PyTorch Data Loader

The data loader delivers small batches to the model during training. Two dataset implementations:

1. **`TokenBlockDataset` (map-style):** loads the tokenized `.pt` file into memory; supports random shuffling. Suitable when the data fits in RAM.
2. **`StreamingTokenBlockDataset` (iterable-style):** reads `cleaned.jsonl` line by line and tokenizes on the fly, never loading the whole dataset into memory (Section 4.3). With multiple workers, each worker reads a disjoint slice of lines.

Both loaders share the same `collate_fn`. How each part connects to training:

| Item | Role in training |
|---|---|
| `labels` and `-100` | `labels` are a copy of `input_ids` (the answer for next-token prediction); padding positions are set to `-100` so the loss ignores them |
| `attention_mask` | Tells the model which positions are padding during the forward pass |
| `BLOCK_SIZE = 512` | Matches the model's maximum sequence length (`n_positions`) |
| `BATCH_SIZE = 8` | Training hyperparameter: number of blocks per training step |
| Shuffling | Mixes different domains within each batch |
| Cast to `int64` | Required by the model's embedding layer and the loss function |

In [18]:
from torch.utils.data import Dataset, IterableDataset, DataLoader


class TokenBlockDataset(Dataset):
    """Map-style dataset wrapping the pre-tokenized .pt file."""

    def __init__(self, tokenized_path):
        data = torch.load(tokenized_path)
        self.input_ids = data["input_ids"]
        self.attention_mask = data["attention_mask"]

    def __len__(self):
        return self.input_ids.shape[0]

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
        }


class StreamingTokenBlockDataset(IterableDataset):
    """Iterable-style dataset: streams documents from JSONL and tokenizes lazily."""

    def __init__(self, jsonl_path, tokenizer_name="gpt2", block_size=512):
        self.jsonl_path = jsonl_path
        self.block_size = block_size
        self.tokenizer_name = tokenizer_name
        self._tokenizer = None  # created lazily in each worker

    def _get_tokenizer(self):
        if self._tokenizer is None:
            self._tokenizer = AutoTokenizer.from_pretrained(self.tokenizer_name)
            if self._tokenizer.pad_token is None:
                self._tokenizer.pad_token = self._tokenizer.eos_token
            self._tokenizer.model_max_length = int(1e12)
        return self._tokenizer

    def __iter__(self):
        tokenizer = self._get_tokenizer()
        pad_id = tokenizer.pad_token_id
        worker_info = torch.utils.data.get_worker_info()
        worker_id = worker_info.id if worker_info else 0
        num_workers = worker_info.num_workers if worker_info else 1

        with open(self.jsonl_path, "r", encoding="utf-8") as f:
            for line_idx, line in enumerate(f):
                if line_idx % num_workers != worker_id:
                    continue
                token_ids = tokenizer.encode(json.loads(line)["text"])
                # Same chunking rule as the map-style pipeline (Section 3)
                for chunk in chunk_tokens(token_ids, self.block_size):
                    n = len(chunk)
                    input_ids = torch.full((self.block_size,), pad_id, dtype=torch.long)
                    input_ids[:n] = torch.tensor(chunk, dtype=torch.long)
                    # Mask by length, not by pad_id: pad_id == eos_id, so a real EOS must stay visible
                    attention_mask = torch.zeros(self.block_size, dtype=torch.long)
                    attention_mask[:n] = 1
                    yield {"input_ids": input_ids, "attention_mask": attention_mask}


def collate_fn(batch):
    input_ids = torch.stack([b["input_ids"] for b in batch]).long()
    attention_mask = torch.stack([b["attention_mask"] for b in batch]).long()
    labels = input_ids.clone()
    labels[attention_mask == 0] = -100
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


def build_map_style_loader(tokenized_path, batch_size=8, shuffle=True, num_workers=2):
    return DataLoader(TokenBlockDataset(tokenized_path), batch_size=batch_size, shuffle=shuffle,
                      num_workers=num_workers, collate_fn=collate_fn, pin_memory=torch.cuda.is_available())


def build_streaming_loader(jsonl_path, tokenizer_name="gpt2", block_size=512, batch_size=8, num_workers=2):
    # Shuffling for an IterableDataset would need a shuffle buffer inside __iter__
    return DataLoader(StreamingTokenBlockDataset(jsonl_path, tokenizer_name, block_size),
                      batch_size=batch_size, shuffle=False, num_workers=num_workers, collate_fn=collate_fn)



In [19]:
def preview(loader, name, n_batches=3):
    print(f"--- {name} ---")
    for i, batch in enumerate(loader):
        print(f"batch {i}: input_ids={tuple(batch['input_ids'].shape)} "
              f"attention_mask={tuple(batch['attention_mask'].shape)} labels={tuple(batch['labels'].shape)}")
        if i + 1 >= n_batches:
            break

preview(build_map_style_loader(TOKENIZED_PATH, batch_size=BATCH_SIZE), "Map-style loader")
preview(build_streaming_loader(CLEANED_PATH, TOKENIZER_NAME, BLOCK_SIZE, batch_size=BATCH_SIZE), "Streaming loader")

--- Map-style loader ---
batch 0: input_ids=(8, 512) attention_mask=(8, 512) labels=(8, 512)
batch 1: input_ids=(8, 512) attention_mask=(8, 512) labels=(8, 512)
batch 2: input_ids=(8, 512) attention_mask=(8, 512) labels=(8, 512)
--- Streaming loader ---
batch 0: input_ids=(8, 512) attention_mask=(8, 512) labels=(8, 512)
batch 1: input_ids=(8, 512) attention_mask=(8, 512) labels=(8, 512)
batch 2: input_ids=(8, 512) attention_mask=(8, 512) labels=(8, 512)


Sanity check: all tensors are `int64`, and every padding position in `labels` is `-100`:

In [20]:
batch = next(iter(build_map_style_loader(TOKENIZED_PATH, batch_size=BATCH_SIZE)))
print(batch["input_ids"].dtype, batch["attention_mask"].dtype, batch["labels"].dtype)
print("labels at pad positions:", batch["labels"][batch["attention_mask"] == 0].unique())

torch.int64 torch.int64 torch.int64
labels at pad positions: tensor([-100])


## 5. Optional Extension: Data Quality Analysis

Section 8 optional extension: per-domain document counts, share of the corpus, word-count statistics, and a length-bucket histogram (word-level, so no tokenizer is needed).

In [21]:
from collections import Counter, defaultdict


def bucket(n_words):
    if n_words < 100:
        return "<100"
    elif n_words < 300:
        return "100-299"
    elif n_words < 700:
        return "300-699"
    elif n_words < 1500:
        return "700-1499"
    return "1500+"


domain_doc_counts = Counter()
word_lengths = defaultdict(list)
length_buckets = defaultdict(Counter)

with open(CLEANED_PATH, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        domain = record.get("domain", "unknown")
        n_words = len(record["text"].split())
        domain_doc_counts[domain] += 1
        word_lengths[domain].append(n_words)
        length_buckets[domain][bucket(n_words)] += 1

total_docs = sum(domain_doc_counts.values())

print(f"{'='*70}\nDOMAIN COVERAGE\n{'='*70}")
print(f"{'Domain':<12}{'Docs':>10}{'% of corpus':>14}")
for domain, count in sorted(domain_doc_counts.items(), key=lambda x: -x[1]):
    print(f"{domain:<12}{count:>10,}{100 * count / total_docs:>13.1f}%")
print(f"{'TOTAL':<12}{total_docs:>10,}{100.0:>13.1f}%")

print(f"\n{'='*70}\nWORD-COUNT STATISTICS PER DOMAIN\n{'='*70}")
print(f"{'Domain':<12}{'Avg words':>12}{'Min':>8}{'Max':>10}")
for domain, lengths in word_lengths.items():
    print(f"{domain:<12}{sum(lengths) / len(lengths):>12.1f}{min(lengths):>8}{max(lengths):>10,}")

print(f"\n{'='*70}\nLENGTH-BUCKET HISTOGRAM (% of each domain's documents)\n{'='*70}")
bucket_order = ["<100", "100-299", "300-699", "700-1499", "1500+"]
print(f"{'Domain':<12}" + "".join(f"{b:>12}" for b in bucket_order))
for domain, n_docs in domain_doc_counts.items():
    print(f"{domain:<12}" + "".join(f"{100 * length_buckets[domain].get(b, 0) / n_docs:>11.1f}%" for b in bucket_order))

DOMAIN COVERAGE
Domain            Docs   % of corpus
wikipedia      165,380         52.2%
news           112,970         35.6%
web             38,714         12.2%
TOTAL          317,064        100.0%

WORD-COUNT STATISTICS PER DOMAIN
Domain         Avg words     Min       Max
news               423.1      50    17,455
web                829.7      81    18,903
wikipedia          737.9      50    34,909

LENGTH-BUCKET HISTOGRAM (% of each domain's documents)
Domain              <100     100-299     300-699    700-1499       1500+
news               10.5%       35.9%       38.0%       13.6%        2.0%
web                 0.0%       24.0%       38.7%       26.0%       11.2%
wikipedia           9.1%       29.8%       31.1%       19.3%       10.7%


## Outputs

| File | Description |
|---|---|
| `data/raw_wikipedia.jsonl`, `raw_news.jsonl`, `raw_web.jsonl` | Raw collected documents per domain |
| `data/collection_manifest.json` | Document count and size per domain |
| `data/cleaned.jsonl` | Cleaned, deduplicated corpus |
| `data/tokenized_blocks.pt` | Full tokenized dataset (`input_ids`, `attention_mask`) |
| `sample_data/sample_dataset.pt` | Small 10-block sample for submission (Section 5.2) |


In [26]:
from google.colab import files
sample = torch.load(SAMPLE_PATH)

for k, v in sample.items():
    print(f"{k}: shape={tuple(v.shape)}, dtype={v.dtype}")
print()

for i in range(sample["input_ids"].shape[0]):
    ids = sample["input_ids"][i]
    mask = sample["attention_mask"][i].bool()
    n_real = int(mask.sum())
    print(f"[block {i}] real tokens: {n_real}, padding: {len(ids) - n_real}")
    print(f"  first 10 token ids: {ids[:10].tolist()}")
    print(f"  text: {tokenizer.decode(ids[mask])[:200]} ...")
    print()

input_ids: shape=(10, 512), dtype=torch.int32
attention_mask: shape=(10, 512), dtype=torch.bool

[block 0] real tokens: 222, padding: 290
  first 10 token ids: [8117, 338, 257, 6452, 14528, 284, 842, 1437, 481, 619]
  text: there's a surprising twist to regina willoughby's last season with columbia city ballet: it's also her 18-year-old daughter melina's first season with the company. regina, 40, will retire from the sta ...

[block 1] real tokens: 247, padding: 265
  first 10 token ids: [1169, 649, 331, 967, 1748, 47735, 3096, 286, 13445, 3414]
  text: the new york city ballet board of directors announced on saturday the interim team that has been appointed to run the artistic side of the company during ballet master in chief peter martins' leave of ...

[block 2] real tokens: 199, padding: 313
  first 10 token ids: [1169, 5206, 8273, 304, 37803, 290, 262, 649, 1786, 1044]
  text: the philadelphia eagles and the new england patriots aren't the only teams bringing super bowl entertainm

## Summary Report

### Pipeline results

| Step | Input | Output | Key numbers |
|---|---|---|---|
| 1. Collection | Wikipedia, CC-News, OpenWebText (streamed from Hugging Face) | `raw_*.jsonl` | 357,171 docs, 1.288 GB (= 1.2 GiB); byte split 60 / 25 / 15 |
| 2. Cleaning | 357,171 docs | `cleaned.jsonl` | 317,064 kept (89%); 26,001 too short, 14,106 exact duplicates |
| 3. Tokenization | 317,064 docs | `tokenized_blocks.pt` | 656,119 blocks × 512 tokens (~2.07 blocks per doc), ~1.6 GB |
| 4. Data loader | `tokenized_blocks.pt` / `cleaned.jsonl` | batches | `(8, 512)` tensors, `int64`, padding labels = `-100` (verified) |

### Key findings

- **Data quality differs strongly by source.** News had the most problems: ~11% of documents that passed the length filter were exact duplicates (syndicated articles), and 12% were too short. Wikipedia had no duplicates but many short stub articles. OpenWebText was almost untouched (3 documents removed) because it was already filtered and deduplicated by its creators.
- **Document share ≠ token share.** By document count the corpus is 52 / 36 / 12 (Wikipedia / news / web), but by training blocks it is 61 / 25 / 14. News articles are short (avg 423 words vs. 738 for Wikipedia and 830 for web), so document counts overstate news's weight in training.
- **The byte-based collection budget carried through.** The final block share (60.8 / 25.2 / 14.0) closely matches the 60 / 25 / 15 target set in Step 1.

### Design decisions

- **Streaming collection** avoids downloading full datasets (Wikipedia alone is 20+ GB).
- **Hashing after cleaning** catches duplicates that differ only in formatting, whitespace, or case.
- **Lowercasing** normalizes text and improves deduplication, at the cost of losing case information (e.g. "Apple" vs. "apple").
- **Compact storage** (`int32` ids, `bool` mask) cuts the file from ~5.4 GB to ~1.6 GB; tensors are cast to `int64` in `collate_fn`.
- **EOS reused as pad** (GPT-2 has no pad token), so the `attention_mask` is built from block length and padding labels are set to `-100`.
- **Two loaders:** map-style for data that fits in RAM (used for training here, supports shuffling); streaming for corpora too large for memory.

### Limitations and future work

- **Exact deduplication only.** Near-duplicates (e.g. the same story with different headers or footers) are not caught; MinHash + LSH would find more, so the 11% news duplicate rate is a lower bound.
- **No within-document deduplication** of repeated paragraphs (e.g. boilerplate).
- **No validation split** yet; a held-out set (e.g. 1%) is needed before training.
- **The streaming loader cannot shuffle**; a shuffle buffer would be needed for training with it.